In [20]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import whisper
import torch
from torch import Tensor
from torch import nn
import torch.nn.init as init

In [3]:
class Conv1d(nn.Conv1d):
    def _conv_forward(
        self, x: Tensor, weight: Tensor, bias: Optional[Tensor]
    ) -> Tensor:
        return super()._conv_forward(
            x, weight.to(x.dtype), None if bias is None else bias.to(x.dtype)
        )

In [46]:
# https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html#conv1d

conv1 = Conv1d(
    in_channels=5,
    out_channels=8,
    kernel_size=3,
    padding=0,
)
# Initialize weights to ones
init.constant_(conv1.weight, 1)

# Optionally, you can also initialize the bias to zeros or another constant if needed
init.constant_(conv1.bias, 0)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [47]:
conv1.weight.shape

torch.Size([8, 5, 3])

In [48]:
conv1.weight

Parameter containing:
tensor([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]], requires_grad=True)

In [49]:
# Define sequence length
sequence_length = 10

# Prepare the input data
input_data = torch.zeros(1, 5, sequence_length)  # Initialize with zeros

# First channel: all ones
input_data[0, 0, :] = 1

# Second channel: increasing sequence
input_data[0, 1, :] = torch.arange(1, sequence_length + 1)

# Third channel: alternating 1s and 0s
input_data[0, 2, :] = torch.tensor([1, 0] * (sequence_length // 2))

# Fourth channel: already all zeros

# Fifth channel: pattern 0, 1, 2, 3, 2, 1, 0, repeating
pattern = torch.tensor([0, 1, 2, 3, 2, 1, 0, 1, 2, 3])
input_data[0, 4, :] = pattern[:sequence_length]

In [50]:
input_data

tensor([[[ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.],
         [ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
         [ 1.,  0.,  1.,  0.,  1.,  0.,  1.,  0.,  1.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  1.,  2.,  3.,  2.,  1.,  0.,  1.,  2.,  3.]]])

In [51]:
extracted_features = conv1(input_data)

In [52]:
extracted_features

tensor([[[14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.],
         [14., 19., 24., 25., 26., 27., 32., 37.]]],
       grad_fn=<ConvolutionBackward0>)

In [53]:
print("Extracted features shape:", extracted_features.shape)

Extracted features shape: torch.Size([1, 8, 8])
